# Class Imbalance in Cheminformatics ML + Augmentation with REINVENT4
### Every Method, Every Trick, and AI-Driven Molecule Generation for Minority Classes

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

## The class imbalance problem in drug discovery

In real-world toxicology and drug discovery datasets, **active / toxic compounds
are always rare**. A typical situation:

```
Ames mutagenicity dataset (Hansen 2009):
  Positive (mutagen):  2,401  compounds  (39%)
  Negative (non-mut):  3,748  compounds  (61%)
  Imbalance ratio:     1.6 : 1  (mild)

hERG cardiotoxicity (literature):
  High risk:    ~15%
  Low risk:     ~85%
  Imbalance:    5.7 : 1  (moderate)

Rare DILI (severe liver injury):
  DILI positive:  ~3-5%
  DILI negative:  ~95-97%
  Imbalance:      20 : 1  (severe)

HTS hit rate:
  Active hits:    0.1-1%
  Inactive:       99-99.9%
  Imbalance:      100-1000 : 1  (extreme)
```

## Why class imbalance breaks naïve ML models

```
Problem: A model that predicts 'inactive' for everything achieves
         99% accuracy on an HTS dataset but is completely useless.

Root cause: Cross-entropy loss treats all samples equally.
            100 majority class samples overpower 1 minority sample.
            The model learns to ignore the rare class.
```

## Complete tutorial map

| Section | Topic | Method |
|---------|-------|--------|
| 1 | Setup and imbalanced dataset | pip install, data |
| 2 | Metrics that actually work | AUC-ROC, PR-AUC, MCC, F1 |
| 3 | Resampling: SMOTE, ROS, RUS | scikit-learn / imbalanced-learn |
| 4 | Algorithm-level fixes | class_weight, focal loss |
| 5 | Ensemble methods | BalancedRF, EasyEnsemble |
| 6 | Threshold tuning | Youden J, precision-recall optimal |
| 7 | REINVENT4: generate minority molecules | Transfer learning + sampling |
| 8 | Integrating generated molecules | Filter, fingerprint, retrain |
| 9 | Full comparison pipeline | All methods benchmarked |
| 10 | Best practices and cheatsheet | Reference card |

---
## Section 1 — Setup and Building an Imbalanced Toxicology Dataset

### pip-only installation

```bash
python -m venv ~/envs/imbalance
source ~/envs/imbalance/bin/activate
pip install --upgrade pip

# Core ML stack
pip install scikit-learn numpy pandas matplotlib seaborn

# Class imbalance toolkit
pip install imbalanced-learn   # SMOTE, ADASYN, RandomOverSampler, etc.

# Cheminformatics
pip install rdkit

# Optional: gradient boosting
pip install xgboost lightgbm

# Kernel
pip install ipykernel
python -m ipykernel install --user --name=imbalance --display-name='Python (imbalance)'
```

### REINVENT4 installation (separate environment recommended)

```bash
# REINVENT4 needs Python 3.10 and specific locked dependencies
python -m venv ~/envs/reinvent4
source ~/envs/reinvent4/bin/activate

# Clone the repository
git clone https://github.com/MolecularAI/REINVENT4.git
cd REINVENT4

# Install with the provided install script
python install.py cpu       # CPU only
# python install.py cu124   # CUDA 12.4 GPU

# Install package itself
pip install --no-deps .

# Verify
reinvent --version
```

In [ ]:
# ── Section 1: Setup, imports, and build an imbalanced Ames dataset ──────────
import warnings, os, json
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

np.random.seed(42)

def chk(name, imp=None):
    try:
        m = __import__(imp or name)
        return f'OK ({getattr(m,"__version__","ok")})'
    except ImportError:
        return 'MISSING — pip install ' + name

print('Package check:')
for n, i in [('sklearn','sklearn'),('imblearn','imblearn'),
              ('rdkit','rdkit'),('xgboost','xgboost'),('lightgbm','lightgbm')]:
    print(f'  {n:15s}: {chk(n, i)}')

os.makedirs('imbalance_output', exist_ok=True)

# ── Build a realistic imbalanced Ames mutagenicity dataset ─────────────────
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, QED, DataStructs

# Representative SMILES with known Ames labels
# Positive (mutagenic) — ICH M7 alerts: nitrosamines, nitro-aromatics, etc.
POSITIVE_SMILES = [
    ('NDMA',              'CN(C)N=O',                    1),
    ('NDEA',              'CCN(CC)N=O',                   1),
    ('NMP',               'CN(C=O)N=O',                   1),
    ('4-NQO',             'O=C(/C=C/c1cccc([N+](=O)[O-])c1)O', 1),
    ('2-AAF',             'CC(=O)Nc1ccc2ccccc2c1',         1),
    ('2-Aminofluorene',   'Nc1ccc2ccccc2c1',              1),
    ('4-aminobiphenyl',   'Nc1ccc(-c2ccccc2)cc1',         1),
    ('Benzidine',         'Nc1ccc(-c2ccc(N)cc2)cc1',      1),
    ('MNNG',              'CN(N=O)C(=N)NC(=N)N',          1),
    ('NNK',               'O=C(CCN=O)c1ccc(N(=O)=O)cc1', 1),
    ('Ethidium_bromide',  'CCc1cc2ccc(N)cc2[n+](CC)c1-c1ccccc1', 1),
    ('Aflatoxin_B1',      'O=c1occc2c1C1C=COC3(OC13)c1c2oc(=O)c1', 1),
    ('Benzo_a_pyrene',    'c1ccc2ccc3cccc4ccc(c1)c2c34', 1),
    ('MeIQ',              'Cc1ccc2nc(N)[nH]c2c1C',        1),
    ('PhIP',              'Cc1ccc(-c2nc3ccccc3[nH]2)cc1', 1),
    ('IQ',                'Cc1ccc2nc(N)[nH]c2c1',         1),
    ('ENU',               'CCN(N=O)C(N)=O',               1),
    ('MNU',               'CN(N=O)C(N)=O',                1),
    ('DMBA',              'Cc1ccc2cc3c(cc2c1)C(C)CC3',    1),
    ('o-Aminoazotoluene',  'Cc1ccccc1N=Nc1ccc(N)c(C)c1', 1),
    ('4-Chloro_o_phenylenediamine', 'Nc1ccc(Cl)cc1N',      1),
    ('Danthron',          'O=C1c2ccccc2C(=O)c2cc(O)ccc21', 1),
    ('Mitomycin_C',       'CO[C@]1(NC(N)=O)[C@@H]2CO[C@H]2CN1[C@@H]1CC1=O', 1),
    ('EMS',               'CCOS(=O)(=O)CC',               1),
    ('DMS',               'COS(=O)(=O)OC',                1),
]

# Negative (non-mutagenic)
NEGATIVE_SMILES = [
    ('Aspirin',       'CC(=O)Oc1ccccc1C(=O)O',         0),
    ('Caffeine',      'Cn1cnc2c1c(=O)n(C)c(=O)n2C',    0),
    ('Paracetamol',   'CC(=O)Nc1ccc(O)cc1',             0),
    ('Ibuprofen',     'CC(C)Cc1ccc(cc1)C(C)C(=O)O',     0),
    ('Metformin',     'CN(C)C(=N)NC(=N)N',              0),
    ('Atorvastatin',  'CC(C)c1c(C(=O)Nc2ccccc2F)c(-c2ccccc2)n(CCC(O)CC(O)CC(=O)O)c1-c1ccc(F)cc1', 0),
    ('Omeprazole',    'COc1ccc2[nH]c([S@@](=O)Cc3ncc(C)c(OC)c3C)nc2c1', 0),
    ('Fluoxetine',    'CNCCC(c1ccccc1)Oc1ccc(cc1)C(F)(F)F', 0),
    ('Simvastatin',   'CCC(C)(C)C(=O)OC1CC(=O)OC2CC(O)CC(CC2=C1)OC(=O)C(C)(C)CC', 0),
    ('Amlodipine',    'CCOC(=O)C1=C(COCCN)NC(C)=C(C(=O)OC)C1c1ccccc1Cl', 0),
    ('Cetirizine',    'OC(=O)CN1CCN(Cc2ccc(Cl)cc2)CC1', 0),
    ('Loratadine',    'CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3ccncc32)CC1', 0),
    ('Metoprolol',    'COCCc1ccc(OCC(O)CNC(C)C)cc1',    0),
    ('Lisinopril',    'OC(=O)C(CCc1ccccc1)NC(CCN1CCCC1C(=O)O)C(=O)O', 0),
    ('Warfarin',      'CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O', 0),
    ('Sildenafil',    'CCCc1nn(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(C)CC4)ccc3OCC)nc12', 0),
    ('Ciprofloxacin', 'OC(=O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O', 0),
    ('Tamoxifen',     'CC/C(=C(\\c1ccccc1)/c1ccc(OCCN(C)C)cc1)c1ccccc1', 0),
    ('Losartan',      'CCCCc1nc(Cl)c(CO)n1Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1', 0),
    ('Pioglitazone',  'O=C1NC(=O)SC1Cc1ccc(OCCCc2ccncc2)cc1', 0),
    ('Gabapentin',    'NCC1(CC(=O)O)CCCCC1',            0),
    ('Pregabalin',    'CC(CN)CC(=O)O',                  0),
    ('Donepezil',     'COc1cc2c(cc1OC)CC1CC(=O)Nc3ccccc3C1=C2', 0),
    ('Memantine',     'CC12CC(CC(C1)(CC(C2)N)C)N',      0),
    ('Oseltamivir',   'CCOC(=O)C1=C[C@@H](OC(CC)CC)[C@H](NC(C)=O)[C@@H](N)C1', 0),
    ('Empagliflozin', 'OC[C@H]1O[C@@H](c2ccc(Cc3ccc(OCC4CCCC4)cc3Cl)cc2)[C@H](O)[C@@H](O)[C@@H]1O', 0),
    ('Metronidazole', 'Cc1ncc([N+](=O)[O-])n1CCO',     0),  # Note: actually +ve in some strains
    ('Saccharin',     'O=C1NS(=O)(=O)c2ccccc21',        0),
    ('Vanillin',      'COc1cc(C=O)ccc1O',               0),
    ('Benzaldehyde',  'O=Cc1ccccc1',                     0),
    ('Benzoic_acid',  'OC(=O)c1ccccc1',                  0),
    ('Acetic_acid',   'CC(=O)O',                         0),
    ('Urea',          'NC(N)=O',                         0),
    ('Glycine',       'NCC(=O)O',                        0),
    ('Adenine',       'Nc1ncnc2[nH]cnc12',              0),
    ('Nicotinamide',  'NC(=O)c1cccnc1',                 0),
    ('Riboflavin',    'Cc1cc2nc3c(=O)[nH]c(=O)nc3n(C[C@H](O)[C@H](O)[C@H](O)CO)c2cc1C', 0),
    ('Thiamine',      'Cc1ncc(C[n+]2csc(CCO)c2C)c(N)n1', 0),
    ('Retinoic_acid', 'CC1=C(/C=C/C(=C/C=C/C(=C/C(=O)O)C)C)CCCC1(C)C', 0),
    ('Sucrose',       'OC[C@H]1O[C@@](CO)(O[C@@H]2O[C@H](CO)[C@@H](O)[C@H](O)[C@H]2O)[C@@H](O)[C@@H]1O', 0),
    ('Cholesterol',   'CC(C)CCC[C@@H](C)[C@H]1CC[C@H]2[C@@H]3CC=C4C[C@@H](O)CC[C@]4(C)[C@H]3CC[C@@]12C', 0),
]

# Build validated dataframe
records = []
for name, smi, label in POSITIVE_SMILES + NEGATIVE_SMILES:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
        arr = np.zeros((2048,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        records.append({'name': name, 'smiles': smi, 'label': label, 'fp': arr})

df_base = pd.DataFrame([{k: v for k, v in r.items() if k != 'fp'} for r in records])
X_base  = np.vstack([r['fp'] for r in records])
y_base  = df_base['label'].values

# Create a more extreme imbalance (realistic HTS scenario)
# Keep all positives, undersample negatives to create 1:4 imbalance
np.random.seed(42)
pos_idx  = np.where(y_base == 1)[0]
neg_idx  = np.where(y_base == 0)[0]
# Augment by adding noise to create 300-compound dataset with 1:4 ratio
np.random.seed(42)
X_aug, y_aug = [], []
for _ in range(6):   # repeat positives 6x with noise
    noise = np.random.normal(0, 0.02, X_base[pos_idx].shape)
    X_aug.append(np.clip(X_base[pos_idx] + noise, 0, 1))
    y_aug.extend([1]*len(pos_idx))
for _ in range(6):   # repeat negatives 6x with noise
    noise = np.random.normal(0, 0.02, X_base[neg_idx].shape)
    X_aug.append(np.clip(X_base[neg_idx] + noise, 0, 1))
    y_aug.extend([0]*len(neg_idx))

X_all = np.vstack(X_aug).astype(np.float32)
y_all = np.array(y_aug)

# Final: 75 positives, 300 negatives = 1:4 imbalance
X_final = np.vstack([X_all[y_all==1][:75], X_all[y_all==0][:300]])
y_final = np.array([1]*75 + [0]*300)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

print('Dataset summary:')
print(f'  Total:  {len(y_final)} compounds')
print(f'  Train:  {len(y_train)} ({sum(y_train==1)} positive, {sum(y_train==0)} negative)')
print(f'  Test:   {len(y_test)}  ({sum(y_test==1)} positive, {sum(y_test==0)} negative)')
print(f'  Imbalance ratio: 1:{int(sum(y_train==0)/sum(y_train==1))}')
print(f'  Fingerprint dim: {X_train.shape[1]}')

---
## Section 2 — Metrics That Actually Matter for Imbalanced Data

**Accuracy is useless on imbalanced datasets.** These metrics are what
pharma teams and computational toxicologists actually report:

| Metric | Formula | What it measures | Best range |
|--------|---------|-----------------|------------|
| **Accuracy** | (TP+TN)/(P+N) | Overall — misleading | — |
| **Sensitivity (recall)** | TP/(TP+FN) | Catches true positives | > 0.80 |
| **Specificity** | TN/(TN+FP) | Avoids false alarms | > 0.70 |
| **Precision (PPV)** | TP/(TP+FP) | Quality of positives | > 0.70 |
| **F1** | 2×P×R/(P+R) | Balance precision/recall | > 0.75 |
| **MCC** | √(TP×TN - FP×FN)/√... | Best single metric | > 0.5 |
| **AUC-ROC** | Area under ROC curve | Discrimination ability | > 0.80 |
| **AUC-PR** | Area under PR curve | Best for extreme imbalance | > 0.60 |

**MCC (Matthews Correlation Coefficient)** is the gold standard single
metric for imbalanced classification — it is the only metric that uses
all four values of the confusion matrix and gives a balanced result even
when class sizes are very different.

In [ ]:
# ── Section 2: All metrics computed and visualised ───────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score,
    matthews_corrcoef, roc_auc_score,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve,
    classification_report,
)

def full_metrics(y_true, y_pred, y_prob, name='Model'):
    """
    Compute the complete set of metrics for imbalanced classification.
    This is the standard reporting format for cheminformatics papers.
    """
    cm   = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    sens = TP / (TP + FN) if (TP + FN) > 0 else 0
    spec = TN / (TN + FP) if (TN + FP) > 0 else 0
    return {
        'name':          name,
        'Accuracy':      accuracy_score(y_true, y_pred),
        'Bal_Accuracy':  balanced_accuracy_score(y_true, y_pred),
        'Sensitivity':   sens,
        'Specificity':   spec,
        'Precision':     precision_score(y_true, y_pred, zero_division=0),
        'F1':            f1_score(y_true, y_pred, zero_division=0),
        'MCC':           matthews_corrcoef(y_true, y_pred),
        'AUC_ROC':       roc_auc_score(y_true, y_prob),
        'AUC_PR':        average_precision_score(y_true, y_prob),
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
    }

# Baseline: naive RF without any imbalance correction
rf_naive = RandomForestClassifier(n_estimators=200, random_state=42)
rf_naive.fit(X_train, y_train)
y_pred_naive = rf_naive.predict(X_test)
y_prob_naive = rf_naive.predict_proba(X_test)[:, 1]

metrics_naive = full_metrics(y_test, y_pred_naive, y_prob_naive, 'Naive RF')

print('Naive Random Forest (no imbalance handling):')
print(f'{"Metric":20s} {"Value":>8}')
print('-'*32)
for k, v in metrics_naive.items():
    if k not in ['name','TP','TN','FP','FN']:
        bar = '█' * int(v * 20) if isinstance(v, float) else ''
        print(f'{k:20s} {v:>8.3f}  {bar}')

print(f'\nConfusion matrix:')
print(f'  TN={metrics_naive["TN"]:3d}  FP={metrics_naive["FP"]:3d}')
print(f'  FN={metrics_naive["FN"]:3d}  TP={metrics_naive["TP"]:3d}')
print()
print('Key insight: Accuracy looks high but sensitivity is low.')
print('The model is biased toward predicting the majority class (non-mutagenic).')

In [ ]:
# ── 2.2 Visualise ROC and PR curves ──────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
BLUE='#1565C0'; RED='#E74C3C'; GREEN='#27AE60'

# Panel 1: Confusion matrix
ax = axes[0]
cm_data = confusion_matrix(y_test, y_pred_naive)
im = ax.imshow(cm_data, cmap='Blues', interpolation='nearest')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Predicted\nNegative','Predicted\nPositive'])
ax.set_yticklabels(['True\nNegative','True\nPositive'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_data[i,j]), ha='center', va='center',
                fontsize=18, fontweight='bold',
                color='white' if cm_data[i,j] > cm_data.max()/2 else 'black')
ax.set_title('Confusion Matrix\n(Naive RF)', fontweight='bold')

# Panel 2: ROC curve
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_prob_naive)
auc_roc = roc_auc_score(y_test, y_prob_naive)
ax.plot(fpr, tpr, BLUE, lw=2.5, label=f'Naive RF (AUC={auc_roc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1.5, alpha=0.5, label='Random baseline')
ax.fill_between(fpr, tpr, alpha=0.1, color=BLUE)
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Precision-Recall curve
ax = axes[2]
prec, rec, _ = precision_recall_curve(y_test, y_prob_naive)
auc_pr = average_precision_score(y_test, y_prob_naive)
baseline_pr = y_test.mean()
ax.plot(rec, prec, RED, lw=2.5, label=f'Naive RF (AUC-PR={auc_pr:.3f})')
ax.axhline(baseline_pr, c='k', lw=1.5, ls='--', alpha=0.5,
           label=f'Random baseline ({baseline_pr:.2f})')
ax.fill_between(rec, prec, alpha=0.1, color=RED)
ax.set_xlabel('Recall (Sensitivity)')
ax.set_ylabel('Precision (PPV)')
ax.set_title('Precision-Recall Curve\n(more informative for imbalanced data)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Baseline Model Evaluation Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_output/metrics_baseline.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: imbalance_output/metrics_baseline.png')

---
## Section 3 — Resampling Methods: SMOTE, ADASYN, ROS, RUS

Resampling methods fix the data distribution **before** training.

### Overview

```
OVERSAMPLING (add synthetic minority samples)
  ROS    RandomOverSampler   — duplicate minority samples with noise
  SMOTE  Synthetic Minority  — interpolate between minority neighbours
  ADASYN Adaptive Synthetic  — focus SMOTE on hard-to-learn regions
  SVMSMOTE                   — SMOTE near SVM decision boundary
  BorderlineSMOTE            — SMOTE near class boundary only

UNDERSAMPLING (remove majority samples)
  RUS    RandomUnderSampler  — randomly remove majority samples
  NearMiss                   — keep majority near minority boundary
  TomekLinks                 — remove boundary majority samples
  EditedNearestNeighbours    — remove noisy majority samples

COMBINED
  SMOTEENN  SMOTE + Edited Nearest Neighbours
  SMOTETomek SMOTE + Tomek links
```

### SMOTE: how it works

```
For each minority sample x_i:
  1. Find k nearest minority neighbours
  2. Randomly pick one neighbour x_j
  3. Create synthetic sample: x_new = x_i + λ × (x_j - x_i)
     where λ ~ Uniform(0, 1)

For fingerprints: interpolation may create non-integer values
→ round to nearest integer or use as soft features
```

### When to use which method

| Ratio | Recommended | Why |
|-------|------------|-----|
| 1:2–5 | SMOTE | Standard, well-validated |
| 1:5–20 | ADASYN or BorderlineSMOTE | Adapts to data topology |
| 1:20–100 | SMOTEENN + class_weight | Combined strategy |
| 1:100+ | ROS + class_weight + focal loss | Most extreme cases |

In [ ]:
# ── Section 3: Resampling methods benchmarked ────────────────────────────────
from imblearn.over_sampling import (
    RandomOverSampler, SMOTE, ADASYN,
    BorderlineSMOTE, SVMSMOTE
)
from imblearn.under_sampling import (
    RandomUnderSampler, NearMiss, TomekLinks,
    EditedNearestNeighbours
)
from imblearn.combine import SMOTEENN, SMOTETomek
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Resamplers to compare
RESAMPLERS = {
    'No resampling':       None,
    'RandomOverSampler':   RandomOverSampler(random_state=42),
    'SMOTE':               SMOTE(random_state=42, k_neighbors=3),
    'BorderlineSMOTE':     BorderlineSMOTE(random_state=42, k_neighbors=3),
    'ADASYN':              ADASYN(random_state=42, n_neighbors=3),
    'RandomUnderSampler':  RandomUnderSampler(random_state=42),
    'TomekLinks':          TomekLinks(),
    'SMOTEENN':            SMOTEENN(random_state=42),
    'SMOTETomek':          SMOTETomek(random_state=42),
}

resample_results = []

for name, resampler in RESAMPLERS.items():
    if resampler is None:
        Xr, yr = X_train.copy(), y_train.copy()
    else:
        try:
            Xr, yr = resampler.fit_resample(X_train, y_train)
        except Exception as e:
            print(f'  {name}: FAILED ({e})')
            continue

    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf.fit(Xr, yr)
    yp  = clf.predict(X_test)
    ypr = clf.predict_proba(X_test)[:, 1]
    m   = full_metrics(y_test, yp, ypr, name)
    m['n_train']   = len(yr)
    m['n_minority'] = int((yr==1).sum())
    m['n_majority'] = int((yr==0).sum())
    resample_results.append(m)
    print(f'{name:25s}: MCC={m["MCC"]:.3f}  AUC-PR={m["AUC_PR"]:.3f}  '
          f'Sens={m["Sensitivity"]:.3f}  Train({m["n_minority"]}+/{m["n_majority"]}-)')

df_resample = pd.DataFrame(resample_results)
print()
print('Best by MCC:', df_resample.loc[df_resample['MCC'].idxmax(), 'name'])
print('Best by AUC-PR:', df_resample.loc[df_resample['AUC_PR'].idxmax(), 'name'])

In [ ]:
# ── 3.2 Visualise resampling comparison ──────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrics_to_plot = ['MCC', 'AUC_PR', 'Sensitivity', 'Specificity', 'F1']
names = df_resample['name'].tolist()
x = np.arange(len(names))
width = 0.15
colours = ['#1565C0','#E74C3C','#27AE60','#E67E22','#8E44AD']

ax = axes[0]
for i, (met, col) in enumerate(zip(metrics_to_plot, colours)):
    vals = df_resample[met].values
    bars = ax.bar(x + i*width, vals, width, color=col, alpha=0.85,
                  label=met, edgecolor='white', lw=0.5)
ax.set_xticks(x + width*2)
ax.set_xticklabels(names, rotation=35, ha='right', fontsize=8)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Resampling Methods Comparison\n(RF classifier)', fontweight='bold')
ax.legend(fontsize=8, loc='upper right'); ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.5, c='k', lw=1, ls='--', alpha=0.3)

# Panel 2: Training set sizes
ax2 = axes[1]
for i, row in df_resample.iterrows():
    ax2.barh(row['name'], row.get('n_majority', 0), color='#1565C0', alpha=0.7)
    ax2.barh(row['name'], row.get('n_minority', 0), left=row.get('n_majority', 0),
             color='#E74C3C', alpha=0.7)
ax2.set_xlabel('Training set size')
ax2.set_title('Training Set Composition After Resampling', fontweight='bold')
import matplotlib.patches as mpatches
ax2.legend(handles=[
    mpatches.Patch(color='#1565C0', alpha=0.7, label='Majority (negative)'),
    mpatches.Patch(color='#E74C3C', alpha=0.7, label='Minority (positive)'),
], fontsize=9)
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle('Resampling Strategy Comparison for Imbalanced Ames Dataset',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_output/resampling_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: imbalance_output/resampling_comparison.png')

---
## Section 4 — Algorithm-Level Fixes: class_weight, Focal Loss, Cost-Sensitive Learning

Instead of changing the data, algorithm-level methods change how the
**loss function** weights errors on each class.

### class_weight parameter

```python
# Automatic: inversely proportional to class frequency
clf = RandomForestClassifier(class_weight='balanced')

# Manual: specify exact weights
clf = RandomForestClassifier(class_weight={0: 1, 1: 4})

# Mathematical: weight_i = n_samples / (n_classes × n_samples_i)
# For our 1:4 dataset:
#   w_positive = 375 / (2 × 75) = 2.5
#   w_negative = 375 / (2 × 300) = 0.625
```

### Focal Loss (Lin et al. 2017)

Focal loss downweights **easy** examples and focuses learning on **hard** ones:

```
Standard cross-entropy:  L = -log(p_t)
Focal loss:              L = -(1 - p_t)^γ × log(p_t)

  γ = 0:   standard cross-entropy
  γ = 2:   recommended default (Lin 2017)
  γ = 5:   very aggressive focus on hard examples

When the model is confident (p_t → 1): (1-p_t)^γ → 0 → loss ≈ 0
When the model is wrong (p_t → 0):     (1-p_t)^γ → 1 → full loss
```

In [ ]:
# ── Section 4: Algorithm-level fixes ─────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    XGB_OK = True
except ImportError:
    XGB_OK = False
try:
    from lightgbm import LGBMClassifier
    LGB_OK = True
except ImportError:
    LGB_OK = False

# ── 4.1 class_weight='balanced' on multiple algorithms ───────────────────────
imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Class imbalance ratio: 1:{imbalance_ratio:.1f}')
print(f'Balanced weights: positive={imbalance_ratio:.2f}x, negative=1.00x')
print()

algo_results = []

# RF with class_weight
for cw in [None, 'balanced', {0:1, 1:4}, {0:1, 1:8}]:
    clf = RandomForestClassifier(n_estimators=200, class_weight=cw,
                                  random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    yp  = clf.predict(X_test)
    ypr = clf.predict_proba(X_test)[:, 1]
    m   = full_metrics(y_test, yp, ypr, f'RF cw={cw}')
    algo_results.append(m)
    print(f'RF class_weight={str(cw):25s}: MCC={m["MCC"]:.3f}  Sens={m["Sensitivity"]:.3f}  AUC-PR={m["AUC_PR"]:.3f}')

# Logistic Regression
for cw in [None, 'balanced']:
    clf = LogisticRegression(class_weight=cw, max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    yp  = clf.predict(X_test)
    ypr = clf.predict_proba(X_test)[:, 1]
    m   = full_metrics(y_test, yp, ypr, f'LR cw={cw}')
    algo_results.append(m)
    print(f'LR class_weight={str(cw):25s}: MCC={m["MCC"]:.3f}  Sens={m["Sensitivity"]:.3f}  AUC-PR={m["AUC_PR"]:.3f}')

# XGBoost with scale_pos_weight
if XGB_OK:
    for spw in [1, imbalance_ratio, imbalance_ratio*2]:
        clf = XGBClassifier(scale_pos_weight=spw, eval_metric='logloss',
                             random_state=42, use_label_encoder=False)
        clf.fit(X_train, y_train)
        yp  = clf.predict(X_test)
        ypr = clf.predict_proba(X_test)[:, 1]
        m   = full_metrics(y_test, yp, ypr, f'XGB spw={spw:.1f}')
        algo_results.append(m)
        print(f'XGB scale_pos_weight={spw:5.1f}          : MCC={m["MCC"]:.3f}  Sens={m["Sensitivity"]:.3f}  AUC-PR={m["AUC_PR"]:.3f}')

df_algo = pd.DataFrame(algo_results)

In [ ]:
# ── 4.2 Focal Loss implementation with PyTorch MLP ───────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al. 2017 — RetinaNet).
    Downweights easy examples, focuses training on hard/rare ones.

    FL(p_t) = -alpha_t × (1 - p_t)^gamma × log(p_t)

    Parameters
    ----------
    alpha : float
        Class weight for positive class (0.25–0.75, default 0.25)
    gamma : float
        Focusing parameter (0=cross-entropy, 2=recommended, 5=aggressive)
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        p       = torch.sigmoid(logits)
        ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t     = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_w = alpha_t * (1 - p_t) ** self.gamma
        return (focal_w * ce_loss).mean()

class MolecularMLP(nn.Module):
    """Simple MLP for binary classification of molecular fingerprints."""
    def __init__(self, input_dim=2048, hidden=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 64),        nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def train_mlp(X_tr, y_tr, X_te, y_te, loss_fn, epochs=100, lr=1e-3):
    Xt = torch.FloatTensor(X_tr); yt = torch.FloatTensor(y_tr)
    model = MolecularMLP(input_dim=X_tr.shape[1])
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    model.train()
    for _ in range(epochs):
        optim.zero_grad()
        loss = loss_fn(model(Xt), yt)
        loss.backward(); optim.step()
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_te))
        probs  = torch.sigmoid(logits).numpy()
    return probs

# Compare loss functions on our dataset
focal_results = []
for gamma, alpha, name in [
    (0.0, 0.5,  'Standard BCE'),
    (1.0, 0.75, 'Focal γ=1 α=0.75'),
    (2.0, 0.75, 'Focal γ=2 α=0.75 (default)'),
    (5.0, 0.80, 'Focal γ=5 α=0.80 (aggressive)'),
]:
    fl = FocalLoss(alpha=alpha, gamma=gamma)
    probs = train_mlp(X_train, y_train, X_test, y_test, fl)
    pred  = (probs > 0.5).astype(int)
    m     = full_metrics(y_test, pred, probs, name)
    focal_results.append(m)
    print(f'{name:35s}: MCC={m["MCC"]:.3f}  Sens={m["Sensitivity"]:.3f}  AUC-PR={m["AUC_PR"]:.3f}')

df_focal = pd.DataFrame(focal_results)

---
## Section 5 — Ensemble Methods Designed for Imbalance

These ensemble algorithms build imbalance-handling directly into the
ensemble construction procedure.

| Method | Idea | Key advantage |
|--------|------|---------------|
| **BalancedRandomForest** | Bootstrap with rebalancing each tree | Robust, fast |
| **EasyEnsemble** | Train AdaBoost on balanced subsets | High sensitivity |
| **BalancedBaggingClassifier** | Bagging with resampling | Flexible base estimator |
| **RUSBoostClassifier** | AdaBoost + RUS each round | Good for extreme imbalance |

**BalancedRandomForest** (Chen et al. 2004) is the industry standard for
imbalanced molecular property prediction — it usually beats SMOTE+RF
because the rebalancing happens inside each bootstrap, better capturing
structural uncertainty.

In [ ]:
# ── Section 5: Balanced ensemble methods ─────────────────────────────────────
from imblearn.ensemble import (
    BalancedRandomForestClassifier,
    EasyEnsembleClassifier,
    BalancedBaggingClassifier,
    RUSBoostClassifier,
)
import numpy as np

ensemble_models = {
    'RF (naive)':                RandomForestClassifier(n_estimators=100, random_state=42),
    'RF (balanced cw)':          RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'BalancedRandomForest':      BalancedRandomForestClassifier(n_estimators=100, random_state=42),
    'EasyEnsemble':              EasyEnsembleClassifier(n_estimators=20, random_state=42),
    'BalancedBagging':           BalancedBaggingClassifier(n_estimators=50, random_state=42),
    'RUSBoost':                  RUSBoostClassifier(n_estimators=100, random_state=42),
}

ensemble_results = []
print(f'{"Model":30s}  {"MCC":>6}  {"AUC-PR":>7}  {"Sens":>6}  {"Spec":>6}  {"F1":>6}')
print('-'*70)
for name, clf in ensemble_models.items():
    clf.fit(X_train, y_train)
    yp  = clf.predict(X_test)
    ypr = clf.predict_proba(X_test)[:, 1]
    m   = full_metrics(y_test, yp, ypr, name)
    ensemble_results.append(m)
    print(f'{name:30s}  {m["MCC"]:>6.3f}  {m["AUC_PR"]:>7.3f}  '
          f'{m["Sensitivity"]:>6.3f}  {m["Specificity"]:>6.3f}  {m["F1"]:>6.3f}')

df_ensemble = pd.DataFrame(ensemble_results)

# Visual comparison
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

metrics_show = ['MCC', 'AUC_PR', 'Sensitivity', 'Specificity', 'F1']
x   = np.arange(len(df_ensemble))
w   = 0.15
cols= ['#1565C0','#E74C3C','#27AE60','#E67E22','#8E44AD']

ax = axes[0]
for i, (met, col) in enumerate(zip(metrics_show, cols)):
    ax.bar(x + i*w, df_ensemble[met], w, color=col, alpha=0.85, label=met, edgecolor='white')
ax.set_xticks(x + w*2)
ax.set_xticklabels(df_ensemble['name'], rotation=35, ha='right', fontsize=8)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Ensemble Methods Comparison', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# Sensitivity vs Specificity tradeoff
ax2 = axes[1]
scatter_cols = ['#95A5A6','#BDC3C7','#E74C3C','#1565C0','#27AE60','#E67E22']
for i, row in df_ensemble.iterrows():
    ax2.scatter(row['Specificity'], row['Sensitivity'], s=200,
                c=scatter_cols[i], edgecolors='k', lw=1.5, zorder=5)
    ax2.annotate(row['name'].split('(')[0].strip(), (row['Specificity'], row['Sensitivity']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
ax2.plot([0,1],[0,1],'k--', lw=1, alpha=0.3)
ax2.axhline(0.8, c='#27AE60', lw=1.5, ls='--', alpha=0.5, label='Target sensitivity 0.8')
ax2.axvline(0.8, c='#1565C0', lw=1.5, ls='--', alpha=0.5, label='Target specificity 0.8')
ax2.fill_between([0.8,1.0],[0.8,0.8],[1.0,1.0], alpha=0.08, color='#27AE60', label='Target zone')
ax2.set_xlabel('Specificity'); ax2.set_ylabel('Sensitivity')
ax2.set_title('Sensitivity vs Specificity Tradeoff', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.suptitle('Ensemble Methods for Imbalanced Classification', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_output/ensemble_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: imbalance_output/ensemble_comparison.png')

---
## Section 6 — Threshold Tuning: Finding the Optimal Decision Boundary

Most classifiers output a probability. The default threshold (0.5) is
**never optimal for imbalanced data**. Moving the threshold changes the
sensitivity/specificity tradeoff.

### Three threshold selection strategies

```
1. Youden J statistic:      argmax(sensitivity + specificity - 1)
   → Maximises overall discriminating ability

2. F1 optimal:              argmax(F1-score)
   → Maximises precision-recall balance

3. Fixed sensitivity:       find threshold where sensitivity = 0.9
   → Regulatory use: ensure 90% of mutagens caught

4. Cost-weighted:           minimize C_FN × FN + C_FP × FP
   → Use when false negatives are much more costly
```

In [ ]:
# ── Section 6: Threshold tuning ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

# Use the best model from Section 5 for threshold tuning demo
brf = BalancedRandomForestClassifier(n_estimators=200, random_state=42)
brf.fit(X_train, y_train)
y_probs = brf.predict_proba(X_test)[:, 1]

# ── Strategy 1: Youden J statistic ───────────────────────────────────────────
fpr, tpr, thresholds_roc = roc_curve(y_test, y_probs)
j_scores   = tpr - fpr  # = sensitivity + specificity - 1
best_j_idx = np.argmax(j_scores)
thresh_youden = thresholds_roc[best_j_idx]

# ── Strategy 2: Optimal F1 ────────────────────────────────────────────────────
prec_arr, rec_arr, thresholds_pr = precision_recall_curve(y_test, y_probs)
f1_arr    = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-8)
best_f1_idx = np.argmax(f1_arr[:-1])
thresh_f1   = thresholds_pr[best_f1_idx]

# ── Strategy 3: Fixed sensitivity = 0.9 ──────────────────────────────────────
target_sens = 0.90
# tpr is sensitivity; find threshold where tpr >= 0.90
valid_idx   = np.where(tpr >= target_sens)[0]
thresh_sens = thresholds_roc[valid_idx[-1]] if len(valid_idx) > 0 else 0.5

# ── Compare threshold strategies ─────────────────────────────────────────────
threshold_strategies = {
    'Default (0.5)':      0.5,
    'Youden J':           thresh_youden,
    f'Optimal F1':        thresh_f1,
    f'Sensitivity=0.9':   thresh_sens,
}

print(f'{"Strategy":25s}  {"Threshold":>10}  {"MCC":>6}  {"Sens":>6}  {"Spec":>6}  {"Prec":>6}  {"F1":>6}')
print('-'*80)
thresh_results = []
for name, thresh in threshold_strategies.items():
    yp = (y_probs >= thresh).astype(int)
    m  = full_metrics(y_test, yp, y_probs, name)
    m['threshold'] = thresh
    thresh_results.append(m)
    print(f'{name:25s}  {thresh:>10.3f}  {m["MCC"]:>6.3f}  {m["Sensitivity"]:>6.3f}  '
          f'{m["Specificity"]:>6.3f}  {m["Precision"]:>6.3f}  {m["F1"]:>6.3f}')

# Visualise threshold tuning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(thresholds_roc, tpr[:-1],      '#27AE60', lw=2.2, label='Sensitivity (TPR)')
ax.plot(thresholds_roc, 1-fpr[:-1],    '#1565C0', lw=2.2, label='Specificity (1-FPR)')
ax.plot(thresholds_roc, j_scores[:-1], '#E74C3C', lw=2.2, ls='--', label='Youden J')
for name, thresh in threshold_strategies.items():
    ax.axvline(thresh, lw=1.5, ls=':', alpha=0.8)
    ax.text(thresh, 0.05, name.split('(')[0][:10], rotation=90, fontsize=7)
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Tuning\nSensitivity vs Specificity', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax2 = axes[1]
t_vals  = np.linspace(0.01, 0.99, 200)
mcc_t   = [matthews_corrcoef(y_test, (y_probs>=t).astype(int)) for t in t_vals]
f1_t    = [f1_score(y_test, (y_probs>=t).astype(int), zero_division=0) for t in t_vals]
ax2.plot(t_vals, mcc_t, '#E74C3C', lw=2.2, label='MCC')
ax2.plot(t_vals, f1_t,  '#1565C0', lw=2.2, label='F1')
ax2.axvline(thresh_youden, c='#27AE60', lw=2, ls='--', label=f'Youden={thresh_youden:.2f}')
ax2.axvline(thresh_f1,     c='#E67E22', lw=2, ls='--', label=f'F1-opt={thresh_f1:.2f}')
ax2.set_xlabel('Decision threshold'); ax2.set_ylabel('Metric value')
ax2.set_title('MCC and F1 vs Threshold', fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.suptitle('Threshold Tuning for Imbalanced Classification', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_output/threshold_tuning.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: imbalance_output/threshold_tuning.png')

---
## Section 7 — REINVENT4: Generating Artificial Minority-Class Molecules

The core insight: instead of synthetic data in fingerprint space (SMOTE),
we can generate **new valid molecules** using a generative AI model focused
on the minority class structure. These are real SMILES that pass physicochemical
filters — not interpolated bit vectors.

### REINVENT4 architecture overview

```
REINVENT4 (AstraZeneca, J. Cheminform. 2024)
   │
   ├── Generators
   │     ├── Reinvent    (RNN, de novo SMILES generation)
   │     ├── LibInvent   (R-group decoration on scaffold)
   │     ├── LinkInvent  (linker design between fragments)
   │     └── Mol2Mol     (transformer, molecule → similar molecule)
   │
   ├── Run modes
   │     ├── sampling          — generate N molecules from a model
   │     ├── scoring           — score SMILES without generation
   │     ├── transfer_learning — fine-tune model on a SMILES set
   │     └── staged_learning   — RL with multi-component scoring
   │
   └── Config: TOML or JSON file
```

### Workflow for minority class augmentation

```
Step 1: Collect minority class SMILES (e.g. 25 Ames-positive compounds)
Step 2: Transfer learning — fine-tune prior on minority class SMILES
        (teaches the model to generate mutagens / hERG actives / etc.)
Step 3: Sampling — generate 500-2000 new SMILES from the fine-tuned model
Step 4: Filter — validity, uniqueness, Ro5, no structural overlap with training
Step 5: Label — keep only high-confidence minority class members
        (score with QSAR model or re-run Ames prediction)
Step 6: Add to training set — retrain ML model on augmented dataset
```

In [ ]:
# ── Section 7: REINVENT4 workflow — install, configure, run ──────────────────
import os

os.makedirs('reinvent4_workspace', exist_ok=True)

# ── 7.1 Installation instructions ────────────────────────────────────────────
print('REINVENT4 Installation (one-time setup)')
print('='*55)
install_steps = '''
# Clone the repository
git clone https://github.com/MolecularAI/REINVENT4.git
cd REINVENT4

# Install via the provided install script (handles PyTorch + CUDA)
python install.py cpu         # CPU-only (for testing)
python install.py cu124       # CUDA 12.4 (recommended for training)

# Install the reinvent package itself
pip install --no-deps .

# Verify
reinvent --version
# > REINVENT 4.x.x

# Download pre-trained prior models from Zenodo
# Models available: reinvent.prior, mol2mol_scaffold_generic.prior,
#                   linkinvent.prior, libinvent.prior
# Reference in config using dot notation: reinvent.prior
# Full list: REINVENT4/reinvent/prior_registry.py
'''
print(install_steps)

# ── 7.2 Save the minority class SMILES file ───────────────────────────────────
# These are our positive class (Ames-positive, ICH M7 alerts)
MINORITY_SMILES = [
    'CN(C)N=O',                # NDMA
    'CCN(CC)N=O',              # NDEA
    'c1cc([N+](=O)[O-])ccc1N', # 4-nitroaniline
    'Nc1ccc2ccccc2c1',          # 2-aminofluorene
    'Nc1ccc(-c2ccccc2)cc1',     # 4-aminobiphenyl
    'c1ccc2ccc3cccc4ccc(c1)c2c34', # benzo[a]pyrene
    'CC(=O)Nc1ccc2ccccc2c1',   # 2-acetylaminofluorene
    'Nc1ccc(-c2ccc(N)cc2)cc1', # benzidine
    'O=C(/C=C/c1cccc([N+](=O)[O-])c1)O', # trans-4-nitrocinnamic acid
    'C1NC1C(=O)O',             # aziridine-2-carboxylic acid (epoxide alert)
    'CCN(N=O)C(N)=O',          # ENU
    'CN(N=O)C(N)=O',           # MNU
    'CCOS(=O)(=O)CC',          # ethyl methanesulfonate
    'COS(=O)(=O)OC',           # dimethyl sulfate
    'c1cncc([N+](=O)[O-])c1',  # 3-nitropyridine
    'O=C1c2ccccc2C(=O)c2cc(O)ccc21', # danthron
    'Cc1ccc2nc(N)[nH]c2c1C',    # MeIQ (cooked food)
    'Cc1ccc(-c2nc3ccccc3[nH]2)cc1', # PhIP
    'CN(N=O)CCO',              # N-nitroso-N-methylethanolamine
    'O=N[N]1CCCC1',            # N-nitrosopyrrolidine
    'O=NN1CCOCC1',             # N-nitrosomorpholine
    'C1CCCN1N=O',              # N-nitrosopiperidine
    'O=NN(CC)CC',              # N-nitrosodiethylamine
    'c1ccc2ncccc2c1',          # quinoline (aromatic amine context)
    'Cc1ccc2nc(N)[nH]c2c1',    # IQ (heterocyclic amine)
]

smiles_path = 'reinvent4_workspace/minority_class.smi'
with open(smiles_path, 'w') as f:
    for smi in MINORITY_SMILES:
        f.write(smi + '\n')
print(f'Saved {len(MINORITY_SMILES)} minority-class SMILES to: {smiles_path}')

In [ ]:
# ── 7.3 Generate REINVENT4 TOML config files ─────────────────────────────────

# ── Config 1: Transfer Learning — fine-tune prior on minority class ───────────
tl_config = '''
# reinvent4_workspace/transfer_learning.toml
# Fine-tune REINVENT prior on Ames-positive (mutagenic) SMILES
# This teaches the model to generate structurally similar compounds

run_type = "transfer_learning"
device   = "cpu"           # change to "cuda:0" if GPU available
tb_logdir = "reinvent4_workspace/tb_TL"

[parameters]
# REINVENT prior: general drug-like molecule generator (RNN)
# Reference using dot notation (auto-downloaded from Zenodo)
input_model_file  = "reinvent.prior"   # or full path: /path/to/reinvent.prior
smiles_file       = "reinvent4_workspace/minority_class.smi"
output_model_file = "reinvent4_workspace/tl_mutagen_model.model"

num_epochs        = 100     # 50-200 for small datasets
save_every_n_epochs = 10    # checkpoint frequency
batch_size        = 16
sample_batch_size = 64      # samples per validation check
num_refs          = 10      # reference molecules for similarity check
'''

# ── Config 2: Sampling — generate new mutagenic compounds ─────────────────────
sampling_config = '''
# reinvent4_workspace/sampling.toml
# Generate new molecules from the fine-tuned model

run_type = "sampling"
device   = "cpu"

[parameters]
model_file       = "reinvent4_workspace/tl_mutagen_model.model"
output_file      = "reinvent4_workspace/generated_mutagens.csv"
num_smiles       = 500         # number of molecules to generate
unique_molecules = true        # deduplicate by canonical SMILES
randomize_smiles = true        # SMILES augmentation for diversity
'''

# ── Config 3: Mol2Mol Transfer Learning — scaffold-based analogues ────────────
mol2mol_tl = '''
# reinvent4_workspace/mol2mol_tl.toml
# Mol2Mol: generate analogues around known mutagenic scaffolds
# Good for: generating structural variants with mutagenic core preserved

run_type = "transfer_learning"
device   = "cpu"
tb_logdir = "reinvent4_workspace/tb_Mol2Mol"

[parameters]
input_model_file  = "mol2mol.prior"    # Mol2Mol transformer prior
smiles_file       = "reinvent4_workspace/minority_class.smi"
output_model_file = "reinvent4_workspace/mol2mol_mutagen.model"

num_epochs        = 100
save_every_n_epochs = 10
batch_size        = 8
sample_batch_size = 64
num_refs          = 10

# Mol2Mol-specific: similarity range for training pairs
pairs.type            = "tanimoto"
pairs.upper_threshold = 0.9      # max similarity to reference
pairs.lower_threshold = 0.5      # min similarity to reference
pairs.min_cardinality = 1
pairs.max_cardinality = 99
'''

# ── Config 4: Staged Learning (RL) — property-guided generation ──────────────
rl_config = '''
# reinvent4_workspace/staged_learning.toml
# Reinforcement learning: generate compounds optimised for:
#   - Mutagenic structural alerts (ICH M7)
#   - Drug-likeness (QED)
#   - Appropriate molecular weight (200-450 Da)

run_type = "staged_learning"
device   = "cpu"
tb_logdir = "reinvent4_workspace/tb_RL"

[parameters]
prior_file = "reinvent.prior"
agent_file = "reinvent4_workspace/tl_mutagen_model.model"

# Stage 1: Learn drug-like mutagens
[[stage]]
max_score    = 0.7      # stop stage when score reaches this
max_steps    = 200      # or when this many steps reached
chkpt_file   = "reinvent4_workspace/rl_stage1.model"
batch_size   = 64

[stage.scoring]
type = "custom_product"

[[stage.scoring.component]]
[[stage.scoring.component.MolecularWeight]]
endpoint.name     = "MW"
endpoint.weight   = 0.3
[[stage.scoring.component.MolecularWeight.endpoint.transform]]
type = "double_sigmoid"
low  = 200
high = 450
coef_div = 20

[[stage.scoring.component]]
[[stage.scoring.component.QED]]
endpoint.name   = "QED_score"
endpoint.weight = 0.4

[[stage.scoring.component]]
[[stage.scoring.component.GroupCount]]
endpoint.name   = "Nitrosamine_alert"
endpoint.weight = 0.3
endpoint.params.smarts = ["[N;!$(N=O)]-N=O", "c[N+](=O)[O-]", "[NH2]c"]
'''

# Save all config files
configs = [
    ('reinvent4_workspace/transfer_learning.toml', tl_config),
    ('reinvent4_workspace/sampling.toml',           sampling_config),
    ('reinvent4_workspace/mol2mol_tl.toml',         mol2mol_tl),
    ('reinvent4_workspace/staged_learning.toml',    rl_config),
]
for path, content in configs:
    with open(path, 'w') as f: f.write(content)
    print(f'Saved: {path}')

print()
print('Run REINVENT4 commands:')
print('  # Step 1: Transfer learning (fine-tune on minority class)')
print('  reinvent -l reinvent4_workspace/tl.log reinvent4_workspace/transfer_learning.toml')
print()
print('  # Step 2: Sample from fine-tuned model')
print('  reinvent -l reinvent4_workspace/sample.log reinvent4_workspace/sampling.toml')
print()
print('  # Step 3: RL-guided generation (optional)')
print('  reinvent -l reinvent4_workspace/rl.log reinvent4_workspace/staged_learning.toml')

---
## Section 8 — Filtering, Validating, and Integrating Generated Molecules

Generated molecules must pass a quality gate before being added to the
training set. Blindly adding all generated SMILES will introduce noise.

### Filtering pipeline

```
REINVENT4 output (CSV with SMILES + NLL)
   │
   ├── Validity check         (RDKit can parse SMILES?)
   ├── Novelty check          (not in training set: Tanimoto < 0.9)
   ├── Chemical filters       (Ro5, TPSA, MW range)
   ├── Alert confirmation     (does it still have the target alerts?)
   ├── QSAR confidence        (predict with existing model, p > 0.7)
   └── Diversity filter       (MaxMin pick unique scaffolds)
```

### How many generated molecules to keep

```
Rule of thumb:
  Generate:  10–50× target addition size
  Pass rate: typically 20–40% after filters
  Target:    1–2× the original minority class size

Example: 75 minority compounds in training set
  Target add:  50–75 validated generated compounds
  Generate:    500 (10×)
  Expected pass: 100–200 (20–40%) → take top 75 by QSAR score
```

In [ ]:
# ── Section 8: Simulate REINVENT4 output and apply filtering pipeline ────────
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.FilterCatalog import FilterCatalogParams, FilterCatalog

# Simulate REINVENT4 generated output CSV
# In production: load the real CSV from reinvent4_workspace/generated_mutagens.csv
np.random.seed(7)

# Simulate 300 generated SMILES (mix of valid/invalid, mutagenic/clean)
GENERATED_POOL = [
    # Valid mutagenic structures (similar to training set)
    'CN(CCC)N=O',              # N-nitrosodiethylamine analogue
    'O=NN(CC)c1ccccc1',         # N-nitrosoaniline derivative
    'Nc1ccc(Cl)cc1',            # 4-chloroaniline (aromatic amine)
    'Nc1ccc(F)cc1',             # 4-fluoroaniline
    'Nc1cccc(F)c1',             # 3-fluoroaniline
    'c1cc(N)ccc1[N+](=O)[O-]', # 4-nitroaniline
    'Nc1ccc([N+](=O)[O-])cc1',  # same
    'c1ccncc1N',                # 4-aminopyridine
    'Nc1cc(C)ccc1C',            # 3,4-dimethylaniline
    'Nc1ccc(OC)cc1',            # 4-methoxyaniline
    'Nc1cc(OC)ccc1',            # 2-methoxyaniline (o-anisidine)
    'CN1CCCN(N=O)CC1',          # N-nitrosopiperazine analogue
    'O=NN1CCCC1',               # N-nitrosopyrrolidine
    'c1ccc2[nH]ccc2c1',         # indole (metabolic activation)
    'Nc1ccc(S(N)(=O)=O)cc1',    # sulfanilamide (borderline)
    'CCOC(=O)N1CCCN(N=O)CC1',   # nitrosamine ester
    'CCC(CC)N=O',               # diethyl nitrosamine
    'O=N(=O)c1ccncc1',          # 3-nitropyridine
    'O=N(=O)c1cccnc1',          # 2-nitropyridine
    'Cc1cc([N+](=O)[O-])ccc1N', # 4-amino-2-nitrotoluene
    # Non-mutagenic (false positives from model)
    'CC(=O)Oc1ccccc1',          # phenyl acetate (aspirin analogue)
    'Oc1ccccc1',                # phenol (negative)
    'CC(O)c1ccccc1',            # 1-phenylethan-1-ol
    'OC(=O)c1ccccc1',           # benzoic acid
    'O=Cc1ccccc1',              # benzaldehyde
    'CN1CCCC1C(=O)O',           # N-methyl proline
    'CC(=O)Nc1ccc(O)cc1',       # paracetamol (non-mutagenic)
    'CC(C)C(=O)Nc1ccc(O)cc1',   # ibuprofen-like fragment
    # Drug-like with alerts (borderline)
    'Cc1ccc(NC(=O)c2ccc([N+](=O)[O-])cc2)cc1', # nitrobenzamide
    'CCOC(=O)c1ccc([N+](=O)[O-])cc1',            # ethyl nitrobenzoate
]

# Build simulated dataframe with NLL scores (lower NLL = more likely from model)
generated_records = []
for i, smi in enumerate(GENERATED_POOL * 5):  # repeat 5x for 150 entries
    mol = Chem.MolFromSmiles(smi)
    if mol:
        nll = np.random.uniform(10, 50)   # negative log-likelihood
        generated_records.append({'smiles': smi, 'NLL': nll})

df_gen = pd.DataFrame(generated_records).drop_duplicates('smiles').reset_index(drop=True)
print(f'Simulated REINVENT4 output: {len(df_gen)} unique SMILES')

# ── Filtering pipeline ────────────────────────────────────────────────────────
ICH_M7_SMARTS = [
    '[N;!$(N=O)]-N=O',    # nitrosamine
    'c[N+](=O)[O-]',       # aromatic nitro
    '[NH2]c',              # primary aromatic amine
    '[$(C=CC=O)]',         # Michael acceptor
    '[C;R0]1OC1',          # acyclic epoxide
]

def filter_generated(df, train_fps, min_tanimoto_novel=0.9):
    results = []
    for _, row in df.iterrows():
        smi = row['smiles']
        mol = Chem.MolFromSmiles(smi)
        if mol is None: continue                    # validity
        mw   = Descriptors.MolWt(mol)
        lp   = Descriptors.MolLogP(mol)
        tpsa = Descriptors.TPSA(mol)
        hbd  = rdMolDescriptors.CalcNumHBD(mol)
        qed  = QED.qed(mol)
        if not (100 < mw < 600): continue           # MW filter
        if not (-2 < lp < 7):    continue           # logP filter
        # Novelty: not too similar to training set
        fp   = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
        max_sim = max(DataStructs.TanimotoSimilarity(fp, tfp) for tfp in train_fps)
        # Alert check: has at least one ICH M7 alert (required for minority class)
        has_alert = any(
            Chem.MolFromSmarts(s) and mol.HasSubstructMatch(Chem.MolFromSmarts(s))
            for s in ICH_M7_SMARTS
        )
        results.append({
            'smiles': smi, 'NLL': row['NLL'],
            'MW': round(mw,1), 'LogP': round(lp,2), 'QED': round(qed,3),
            'max_sim_train': round(max_sim,3),
            'has_alert': has_alert,
            'novel': max_sim < min_tanimoto_novel,
        })
    return pd.DataFrame(results)

# Build fingerprints from positive training set
train_fps = [
    AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), 2, 2048)
    for s in MINORITY_SMILES
]

df_filtered = filter_generated(df_gen, train_fps)
df_accepted = df_filtered[(df_filtered['has_alert']) & (df_filtered['novel'])]

print(f'Generated: {len(df_gen)}')
print(f'After filters: {len(df_filtered)}')
print(f'Has alert + novel: {len(df_accepted)}')
print()
print('Accepted molecules (sample):')
print(df_accepted[['smiles','MW','LogP','QED','max_sim_train']].head(8).to_string(index=False))

df_accepted.to_csv('reinvent4_workspace/filtered_generated.csv', index=False)
print('\nSaved: reinvent4_workspace/filtered_generated.csv')

In [ ]:
# ── 8.2 Retrain classifier with generated molecules added ─────────────────────
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Build fingerprint matrix for accepted generated molecules
gen_fps_list = []
for smi in df_accepted['smiles']:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp  = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
        arr = np.zeros((2048,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        gen_fps_list.append(arr)

X_gen   = np.vstack(gen_fps_list)
y_gen   = np.ones(len(X_gen), dtype=int)   # all generated = positive (label 1)

# Augmented training set
X_aug_train = np.vstack([X_train, X_gen])
y_aug_train = np.concatenate([y_train, y_gen])

print('Augmented training set:')
print(f'  Original: {len(y_train)} ({(y_train==1).sum()} pos, {(y_train==0).sum()} neg)')
print(f'  Added:    {len(y_gen)} generated positives')
print(f'  Final:    {len(y_aug_train)} ({(y_aug_train==1).sum()} pos, {(y_aug_train==0).sum()} neg)')
print(f'  New ratio: 1:{(y_aug_train==0).sum()/(y_aug_train==1).sum():.1f}')

# Compare baseline vs augmented
results_aug = {}
for label, Xtr, ytr in [('Without augmentation', X_train, y_train),
                          ('With REINVENT4 augmentation', X_aug_train, y_aug_train)]:
    clf = BalancedRandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(Xtr, ytr)
    yp  = clf.predict(X_test)
    ypr = clf.predict_proba(X_test)[:, 1]
    m   = full_metrics(y_test, yp, ypr, label)
    results_aug[label] = m
    print(f'{label}:')
    print(f'  MCC={m["MCC"]:.3f}  AUC-PR={m["AUC_PR"]:.3f}  Sensitivity={m["Sensitivity"]:.3f}')
    print()

---
## Section 9 — Complete Benchmark: All Methods Head-to-Head

This section benchmarks every strategy from this tutorial on the same
dataset so you can see the relative impact of each approach.

In [ ]:
# ── Section 9: Complete benchmark of all imbalance strategies ───────────────
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
from sklearn.ensemble import RandomForestClassifier
import numpy as np, pandas as pd

# Run all strategies
all_strategies = []

# 1. Naive RF (baseline)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'Naive RF (baseline)'))

# 2. RF + class_weight=balanced
clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'RF + class_weight=balanced'))

# 3. SMOTE + RF
from imblearn.over_sampling import SMOTE
Xr, yr = SMOTE(random_state=42, k_neighbors=3).fit_resample(X_train, y_train)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xr, yr)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'SMOTE + RF'))

# 4. ADASYN + RF
from imblearn.over_sampling import ADASYN
Xr, yr = ADASYN(random_state=42, n_neighbors=3).fit_resample(X_train, y_train)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xr, yr)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'ADASYN + RF'))

# 5. SMOTEENN + RF
from imblearn.combine import SMOTEENN
Xr, yr = SMOTEENN(random_state=42).fit_resample(X_train, y_train)
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xr, yr)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'SMOTEENN + RF'))

# 6. BalancedRandomForest
clf = BalancedRandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'BalancedRandomForest'))

# 7. EasyEnsemble
clf = EasyEnsembleClassifier(n_estimators=20, random_state=42)
clf.fit(X_train, y_train)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'EasyEnsemble'))

# 8. BalancedRF + threshold tuning
clf = BalancedRandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)
ypr = clf.predict_proba(X_test)[:,1]
from sklearn.metrics import roc_curve
fpr_t, tpr_t, thresh_t = roc_curve(y_test, ypr)
j    = tpr_t - fpr_t
best = thresh_t[np.argmax(j)]
yp   = (ypr >= best).astype(int)
all_strategies.append(full_metrics(y_test, yp, ypr, 'BalRF + Youden threshold'))

# 9. REINVENT4 augmentation + BalancedRF
clf = BalancedRandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_aug_train, y_aug_train)
yp  = clf.predict(X_test)
ypr = clf.predict_proba(X_test)[:,1]
all_strategies.append(full_metrics(y_test, yp, ypr, 'REINVENT4 aug + BalRF'))

# 10. Best combined: SMOTE + BalancedRF + threshold
Xr, yr = SMOTE(random_state=42, k_neighbors=3).fit_resample(X_aug_train, y_aug_train)
clf = BalancedRandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(Xr, yr)
ypr = clf.predict_proba(X_test)[:,1]
fpr_t, tpr_t, thresh_t = roc_curve(y_test, ypr)
best = thresh_t[np.argmax(tpr_t - fpr_t)]
yp   = (ypr >= best).astype(int)
all_strategies.append(full_metrics(y_test, yp, ypr, 'REINVENT4+SMOTE+BalRF+thresh'))

df_all = pd.DataFrame(all_strategies)
print('Complete benchmark results:')
print(f'{"Strategy":35s}  {"MCC":>6}  {"AUC-PR":>7}  {"Sens":>6}  {"Spec":>6}  {"F1":>6}')
print('-'*80)
for _, row in df_all.sort_values('MCC', ascending=False).iterrows():
    print(f'{row["name"]:35s}  {row["MCC"]:>6.3f}  {row["AUC_PR"]:>7.3f}  '
          f'{row["Sensitivity"]:>6.3f}  {row["Specificity"]:>6.3f}  {row["F1"]:>6.3f}')

In [ ]:
# ── 9.2 Final benchmark visualisation ───────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
BLUE='#1565C0'; RED='#E74C3C'; GREEN='#27AE60'; GOLD='#F1C40F'

df_sorted = df_all.sort_values('MCC', ascending=True).reset_index(drop=True)

# Panel 1: Horizontal bar chart of MCC
ax = axes[0]
cols = [GREEN if 'REINVENT' in n else BLUE if 'Balanced' in n or 'Easy' in n
        else RED for n in df_sorted['name']]
ax.barh(df_sorted['name'], df_sorted['MCC'], color=cols, alpha=0.85, edgecolor='white')
ax.axvline(0.5, c='k', lw=2, ls='--', alpha=0.5, label='MCC=0.5 (good threshold)')
for i, (name, val) in enumerate(zip(df_sorted['name'], df_sorted['MCC'])):
    ax.text(val+0.005, i, f'{val:.3f}', va='center', fontsize=8.5, fontweight='bold')
ax.set_xlabel('Matthews Correlation Coefficient (MCC)')
ax.set_title('All Methods: MCC Score\n(higher = better)', fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color=GREEN, alpha=0.85, label='REINVENT4 augmentation'),
    mpatches.Patch(color=BLUE,  alpha=0.85, label='Ensemble methods'),
    mpatches.Patch(color=RED,   alpha=0.85, label='Other methods'),
], fontsize=9)
ax.grid(True, alpha=0.3, axis='x')

# Panel 2: Radar chart — Sens/Spec/F1/MCC/AUC-PR
metrics_radar = ['Sensitivity','Specificity','F1','MCC','AUC_PR']
n = len(metrics_radar)
angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist() + [0]
ax2 = fig.add_subplot(1, 2, 2, projection='polar')
highlight = ['Naive RF (baseline)', 'BalancedRandomForest', 'REINVENT4+SMOTE+BalRF+thresh']
radar_cols = ['#BDC3C7', '#1565C0', '#27AE60']
for mname, col in zip(highlight, radar_cols):
    row = df_all[df_all['name'] == mname].iloc[0]
    vals = [row[m] for m in metrics_radar] + [row[metrics_radar[0]]]
    ax2.plot(angles, vals, color=col, lw=2.5, label=mname.split('(')[0][:25])
    ax2.fill(angles, vals, color=col, alpha=0.1)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(metrics_radar, fontsize=9)
ax2.set_ylim(0, 1)
ax2.set_title('Radar Chart: Key Metrics\n(outer = better)', fontweight='bold', pad=20)
ax2.legend(fontsize=8, loc='upper right')

plt.suptitle('Complete Class Imbalance Benchmark: All Strategies',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('imbalance_output/complete_benchmark.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: imbalance_output/complete_benchmark.png')

---
## Section 10 — Best Practices, Decision Guide, and Complete Cheatsheet

### Decision guide: which method to choose

```
Imbalance ratio  │  Dataset size  │  Recommended strategy
─────────────────┼────────────────┼────────────────────────────────────────
1:2 – 1:5        │  Any           │  class_weight='balanced' + RF or GBM
1:5 – 1:20       │  >500 samples  │  SMOTE + BalancedRandomForest
1:5 – 1:20       │  <500 samples  │  ADASYN + threshold tuning
1:20 – 1:100     │  >1000         │  EasyEnsemble or REINVENT4 + BalRF
1:20 – 1:100     │  <1000         │  REINVENT4 augmentation + SMOTEENN
>1:100           │  Any           │  REINVENT4 + Focal loss + Youden thresh
─────────────────┴────────────────┴────────────────────────────────────────
```

### REINVENT4 mode selection

| You have | REINVENT4 mode | Goal |
|----------|---------------|------|
| 10–50 minority SMILES | `transfer_learning` + `sampling` | Generate close analogues |
| 50+ minority SMILES | `staged_learning` (RL) | Property-guided generation |
| Core scaffold + want R-group diversity | `libinvent` | Scaffold decoration |
| Two fragments + want linker | `linkinvent` | Linker design |
| Reference molecule + want analogues | `mol2mol` sampling | Structural analogues |

In [ ]:
# ── Section 10: Complete cheatsheet + file listing ───────────────────────────
import os

cheat = [
    'CLASS IMBALANCE IN CHEMINFORMATICS ML — COMPLETE REFERENCE',
    '',
    'PIP INSTALL',
    '  pip install scikit-learn imbalanced-learn xgboost lightgbm',
    '  pip install rdkit torch numpy pandas matplotlib seaborn',
    '',
    'METRICS — ALWAYS REPORT THESE (never just accuracy)',
    '  MCC:         matthews_corrcoef(y_true, y_pred)          # best single metric',
    '  AUC-PR:      average_precision_score(y_true, y_prob)    # for extreme imbalance',
    '  AUC-ROC:     roc_auc_score(y_true, y_prob)',
    '  Sensitivity: recall_score(y_true, y_pred)               # TPR',
    '  Specificity: TN / (TN + FP)                             # TNR',
    '',
    'RESAMPLING QUICK REFERENCE',
    '  from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler',
    '  from imblearn.under_sampling import RandomUnderSampler, TomekLinks',
    '  from imblearn.combine import SMOTEENN, SMOTETomek',
    '  X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)',
    '',
    'ALGORITHM-LEVEL FIXES',
    '  RF:    RandomForestClassifier(class_weight="balanced")',
    '  XGB:   XGBClassifier(scale_pos_weight=imbalance_ratio)',
    '  LGB:   LGBMClassifier(class_weight="balanced", is_unbalance=True)',
    '  LR:    LogisticRegression(class_weight="balanced")',
    '  SVM:   SVC(class_weight="balanced", probability=True)',
    '',
    'FOCAL LOSS (PyTorch)',
    '  FL = -(1-p_t)^gamma * log(p_t)',
    '  gamma=0: standard CE  |  gamma=2: recommended  |  gamma=5: aggressive',
    '  alpha: positive class weight (0.25-0.75)',
    '',
    'ENSEMBLE METHODS',
    '  from imblearn.ensemble import BalancedRandomForestClassifier',
    '  from imblearn.ensemble import EasyEnsembleClassifier',
    '  from imblearn.ensemble import RUSBoostClassifier',
    '  BalancedRandomForestClassifier(n_estimators=200, random_state=42)',
    '',
    'THRESHOLD TUNING',
    '  # Youden J (optimal sensitivity + specificity)',
    '  fpr, tpr, thresholds = roc_curve(y_test, y_prob)',
    '  best_thresh = thresholds[np.argmax(tpr - fpr)]',
    '  y_pred = (y_prob >= best_thresh).astype(int)',
    '',
    'REINVENT4 AUGMENTATION WORKFLOW',
    '  1. git clone https://github.com/MolecularAI/REINVENT4.git',
    '     cd REINVENT4 && python install.py cpu && pip install --no-deps .',
    '  2. Write minority SMILES to .smi file (one per line)',
    '  3. Transfer learning TOML: run_type="transfer_learning"',
    '     input_model_file="reinvent.prior" smiles_file="minority.smi"',
    '  4. Run: reinvent -l tl.log transfer_learning.toml',
    '  5. Sampling TOML: run_type="sampling" num_smiles=500',
    '  6. Filter: validity + novelty (Tc<0.9) + alert check + QSAR score',
    '  7. Retrain model on original + accepted generated molecules',
    '',
    'DECISION GUIDE',
    '  1:2–5 ratio:    class_weight=balanced + RF',
    '  1:5–20 ratio:   SMOTE + BalancedRandomForest + threshold tuning',
    '  1:20–100 ratio: EasyEnsemble OR REINVENT4 + BalRF',
    '  >1:100 ratio:   REINVENT4 + Focal loss + Youden threshold',
    '',
    'REINVENT4 MODE SELECTION',
    '  10-50 minority SMILES:  transfer_learning + sampling',
    '  50+ minority SMILES:    staged_learning (RL, property-guided)',
    '  Scaffold decoration:    libinvent',
    '  Linker design:          linkinvent',
    '  Structural analogues:   mol2mol',
]
print('\n'.join(cheat))

print()
print('Files created:')
print('='*60)
files = [
    ('imbalance_output/metrics_baseline.png',   'Baseline metrics'),
    ('imbalance_output/resampling_comparison.png','Resampling methods'),
    ('imbalance_output/ensemble_comparison.png', 'Ensemble methods'),
    ('imbalance_output/threshold_tuning.png',    'Threshold tuning'),
    ('imbalance_output/complete_benchmark.png',  'Final benchmark'),
    ('reinvent4_workspace/minority_class.smi',   'REINVENT4 input SMILES'),
    ('reinvent4_workspace/transfer_learning.toml','REINVENT4 TL config'),
    ('reinvent4_workspace/sampling.toml',         'REINVENT4 sampling config'),
    ('reinvent4_workspace/staged_learning.toml',  'REINVENT4 RL config'),
    ('reinvent4_workspace/mol2mol_tl.toml',       'REINVENT4 Mol2Mol config'),
    ('reinvent4_workspace/filtered_generated.csv','Filtered generated molecules'),
]
for path, desc in files:
    status = 'OK' if os.path.exists(path) else '--'
    print(f'  [{status}] {path:45s} {desc}')